# Z2005 — Binary Trees

Binary trees are the first data structure in this course where the *shape* of
the structure, not just its contents, determines performance. This notebook
builds trees from a plain node class up through traversals, binary search
trees, AVL balancing, and the ordered-set operations trees make possible.
Work through it top to bottom, run every cell, and complete the exercises
before checking the solutions at the end.

## Learning Objectives

By the end of this notebook you will be able to:

- Implement a binary tree node class and build small trees by hand in code.
- Write preorder, inorder, and postorder traversals both recursively and
  iteratively using an explicit stack.
- Implement level-order (breadth-first) traversal using a queue.
- Implement BST search, insertion, and deletion, including the two-children
  deletion case using an in-order successor.
- Explain why BST operations are O(h) and why inserting sorted data produces
  a degenerate, list-like tree with O(n) operations.
- Explain, at a conceptual level, how AVL and red-black trees use rotations
  or recoloring to guarantee O(log n) height, and implement AVL insertion
  with rotations.

## How to use this notebook

Run the cells in order, top to bottom. Cells marked `TODO` are yours to
complete — replace the `raise NotImplementedError` with working code. Every
exercise is followed by a self-check cell built from `assert` statements: it
raises an `AssertionError` (or prints nothing helpful) if your code is wrong,
and prints a `✅` message if it is correct. The `## Solutions` section at the
very end has fully worked answers — try each exercise yourself first.

## 1. The tree node and building a tree by hand

A binary tree is built from nodes, and each node is almost embarrassingly
simple: it holds a value and two references, `left` and `right`, either of
which may be `None`. Everything else — traversal order, search efficiency,
balance — is a *consequence* of how those references are wired together, not
something the node itself enforces. This is the same idea as a linked list
node with one extra pointer, and it is worth noticing that a tree is really
just "a linked list that is allowed to branch."

A common early mistake is to think a tree needs a special "tree" class that
owns all the nodes. It does not: the tree *is* its root node, plus whatever
child references you follow from there. Losing the reference to the root
means losing the entire tree, exactly as losing the head of a linked list
loses the whole list.

In [ ]:
class TreeNode:
    """A single node in a binary tree: one value, two children."""

    def __init__(self, value):
        self.value = value
        self.left = None   # reference to left child TreeNode, or None
        self.right = None  # reference to right child TreeNode, or None

    def __repr__(self):
        return f"TreeNode({self.value!r})"


# Build this tree by hand, wiring up references directly:
#
#            8
#          /   \
#         3     10
#        / \      \
#       1   6      14
#
root = TreeNode(8)
root.left = TreeNode(3)
root.right = TreeNode(10)
root.left.left = TreeNode(1)
root.left.right = TreeNode(6)
root.right.right = TreeNode(14)

# Sanity check: the root's left grandchild-left is 1, right subtree's right is 14
assert root.left.left.value == 1
assert root.right.right.value == 14
print("Tree built:", root, "with left child", root.left, "and right child", root.right)

## 2. Depth-first traversals: preorder, inorder, postorder

There are three natural orders to visit a binary tree if you always go
"down" before "across": visit the node itself either before, between, or
after visiting its two subtrees. That single choice gives preorder
(visit, left, right), inorder (left, visit, right), and postorder
(left, right, visit). Inorder is the one to remember specially: for a binary
*search* tree it produces values in sorted order, which is the whole reason
BSTs are useful for anything beyond lookup.

The recursive versions are almost a direct transcription of the definitions
above, which is exactly why they are worth writing out slowly the first time:
the recursion mirrors the tree's own recursive structure (a tree is a node
plus two smaller trees). The pitfall to watch for is forgetting the base
case — every recursive traversal must do nothing when `node is None`,
otherwise it crashes on the first leaf's missing child.

In [ ]:
def preorder_recursive(node, result=None):
    """Visit node, then left subtree, then right subtree."""
    if result is None:
        result = []
    if node is not None:            # base case: nothing to do past a leaf
        result.append(node.value)   # visit BEFORE recursing = preorder
        preorder_recursive(node.left, result)
        preorder_recursive(node.right, result)
    return result


def inorder_recursive(node, result=None):
    """Visit left subtree, then node, then right subtree."""
    if result is None:
        result = []
    if node is not None:
        inorder_recursive(node.left, result)
        result.append(node.value)   # visit BETWEEN the two recursive calls
        inorder_recursive(node.right, result)
    return result


def postorder_recursive(node, result=None):
    """Visit left subtree, then right subtree, then node."""
    if result is None:
        result = []
    if node is not None:
        postorder_recursive(node.left, result)
        postorder_recursive(node.right, result)
        result.append(node.value)   # visit AFTER both recursive calls = postorder
    return result


print("preorder: ", preorder_recursive(root))   # visit-first order
print("inorder:  ", inorder_recursive(root))    # sorted order for a BST
print("postorder:", postorder_recursive(root))  # visit-last order
assert preorder_recursive(root) == [8, 3, 1, 6, 10, 14]
assert inorder_recursive(root) == [1, 3, 6, 8, 10, 14]
assert postorder_recursive(root) == [1, 6, 3, 14, 10, 8]
print("Recursive traversal checks passed")

### Iterative traversals with an explicit stack

Recursion works because Python is silently maintaining a call stack for you.
You can maintain that stack yourself with an explicit `list` used as a
stack (`append` to push, `pop` to pop), which is worth doing at least once:
it is what the recursive version is doing under the hood, and it is the
technique you need when recursion depth becomes a real constraint (a
degenerate tree of 10,000 nodes will blow Python's recursion limit, but an
explicit stack on the heap will not).

Preorder is the easiest to make iterative because "visit, then push right,
then push left" mirrors the recursive order directly (push right *before*
left so that left is popped and processed first). Inorder is the fiddly one:
you have to walk all the way left *before* you are allowed to visit
anything, which means the loop needs two phases — descend left while
pushing, then visit and step right.

In [ ]:
def preorder_iterative(node):
    """Preorder traversal using an explicit stack instead of recursion."""
    if node is None:
        return []
    result = []
    stack = [node]
    while stack:
        current = stack.pop()
        result.append(current.value)     # visit when popped
        if current.right is not None:
            stack.append(current.right)  # push right FIRST
        if current.left is not None:
            stack.append(current.left)   # so left is popped and visited first
    return result


def inorder_iterative(node):
    """Inorder traversal using an explicit stack: descend left, then visit."""
    result = []
    stack = []
    current = node
    while stack or current is not None:
        while current is not None:       # phase 1: go as far left as possible,
            stack.append(current)        # remembering every node on the way down
            current = current.left
        current = stack.pop()            # phase 2: nothing further left, so visit
        result.append(current.value)
        current = current.right          # then move to the right subtree
    return result


assert preorder_iterative(root) == preorder_recursive(root)
assert inorder_iterative(root) == inorder_recursive(root)
print("preorder (iterative):", preorder_iterative(root))
print("inorder (iterative): ", inorder_iterative(root))
print("Iterative traversals match their recursive counterparts")

## 3. Level-order traversal (breadth-first, with a queue)

All three traversals above go deep before they go wide: they follow one
branch all the way down before backtracking. Level-order traversal does the
opposite — it visits the tree row by row, top to bottom, left to right
within a row — which is exactly what you want when printing a tree for a
human to read, or when searching for the *shallowest* match. The standard
tool for "process in the order discovered, one layer at a time" is a queue
(first-in, first-out), and Python's `collections.deque` gives an O(1)
append/popleft queue rather than the O(n) pops you would get from using a
plain list as a queue.

In [ ]:
from collections import deque

def level_order(root_node):
    """Breadth-first traversal: visit level by level using a FIFO queue."""
    if root_node is None:
        return []
    result = []
    queue = deque([root_node])   # deque.popleft() is O(1); list.pop(0) is O(n)
    while queue:
        node = queue.popleft()   # dequeue the next node to visit, in FIFO order
        result.append(node.value)
        if node.left is not None:
            queue.append(node.left)    # enqueue children left-to-right
        if node.right is not None:
            queue.append(node.right)
    return result


print("level-order:", level_order(root))
assert level_order(root) == [8, 3, 10, 1, 6, 14]
print("Level-order traversal check passed")

## 4. Binary search trees: search, insert, delete

A binary search tree (BST) is a plain binary tree with one extra rule
enforced at every single node: everything in the left subtree is smaller
than the node's value, and everything in the right subtree is larger. That
one invariant is what makes search fast — at each node you can throw away
an entire subtree just by comparing against the current value, the same way
binary search on a sorted array discards half the remaining elements each
step.

A subtle but common bug is checking only the immediate children against the
BST property instead of the *whole* subtree — a node can be locally "in
order" relative to its parent while still violating the BST property
relative to a grandparent further up. `is_valid_bst` below carries a
`(low, high)` bound down the recursion specifically to catch that case.

Deletion is the operation students usually find hardest, because it has
three genuinely different cases: deleting a leaf (just remove it), deleting
a node with one child (splice the child up into the parent's slot), and
deleting a node with two children (the hard case — you cannot just remove
it without losing one of its subtrees, so instead you copy in its in-order
successor's value and delete the successor instead, which is guaranteed to
have at most one child).

In [ ]:
def bst_search(node, target):
    """Return the TreeNode holding target, or None if it isn't present."""
    if node is None or node.value == target:
        return node                          # found it, or ran off the tree
    if target < node.value:
        return bst_search(node.left, target)  # only the left subtree can hold it
    return bst_search(node.right, target)     # only the right subtree can hold it


def bst_insert(node, value):
    """Insert value into the BST rooted at node; return the (possibly new) root."""
    if node is None:
        return TreeNode(value)          # found the empty slot: this is where it goes
    if value < node.value:
        node.left = bst_insert(node.left, value)
    elif value > node.value:
        node.right = bst_insert(node.right, value)
    # if value == node.value we do nothing: no duplicates in this BST
    return node


def bst_delete(node, value):
    """Delete value from the BST rooted at node; return the (possibly new) root."""
    if node is None:
        return None                     # value not found, nothing to do
    if value < node.value:
        node.left = bst_delete(node.left, value)
    elif value > node.value:
        node.right = bst_delete(node.right, value)
    else:
        # node.value == value: this is the node to remove
        if node.left is None and node.right is None:
            return None                 # case 1: leaf, just drop it
        if node.left is None:
            return node.right           # case 2: one child, splice it up
        if node.right is None:
            return node.left            # case 2: one child (mirror), splice it up
        # case 3: two children — find the in-order successor (smallest value
        # in the right subtree), copy its value here, then delete it there
        successor = node.right
        while successor.left is not None:
            successor = successor.left
        node.value = successor.value
        node.right = bst_delete(node.right, successor.value)
    return node


def is_valid_bst(node, low=float("-inf"), high=float("inf")):
    """Check the BST property holds for the WHOLE subtree, not just parent/child."""
    if node is None:
        return True
    if not (low < node.value < high):
        return False
    return (is_valid_bst(node.left, low, node.value)
            and is_valid_bst(node.right, node.value, high))


bst_root = None
for v in [5, 3, 8, 1, 4, 7, 9]:
    bst_root = bst_insert(bst_root, v)

assert inorder_recursive(bst_root) == [1, 3, 4, 5, 7, 8, 9]  # inorder = sorted!
assert is_valid_bst(bst_root)
assert bst_search(bst_root, 7) is not None
assert bst_search(bst_root, 100) is None

bst_root = bst_delete(bst_root, 5)   # deletes the root, a two-children case
assert is_valid_bst(bst_root)
assert inorder_recursive(bst_root) == [1, 3, 4, 7, 8, 9]
print("BST search/insert/delete checks passed")

## 5. BST complexity and worst-case degeneration

Every BST operation above walks a single root-to-leaf path, so its cost is
proportional to the tree's *height* (h), not its number of nodes (n). If the
tree stays roughly balanced, h is about log2(n), and search/insert/delete
are all O(log n) — fast even for millions of nodes. But nothing in the code
above enforces balance. If you insert values that already arrive in sorted
order, every new node becomes the right child of the previous one, and the
"tree" degenerates into what is structurally a linked list, with h = n and
every operation O(n).

The `timeit` comparison below demonstrates this live rather than just
asserting it: searching a tree built from sorted input should be visibly
slower than searching one built from shuffled input of the same size,
because the degenerate tree forces a linear scan instead of a logarithmic
one.

In [ ]:
import random
import timeit

def tree_height(node):
    """Height of a tree: -1 for empty, 0 for a single leaf, else 1 + taller subtree."""
    if node is None:
        return -1
    return 1 + max(tree_height(node.left), tree_height(node.right))


n = 500
sorted_values = list(range(n))
shuffled_values = list(range(n))
random.seed(42)
random.shuffle(shuffled_values)

degenerate_root = None
for v in sorted_values:                 # inserting in sorted order...
    degenerate_root = bst_insert(degenerate_root, v)

balanced_ish_root = None
for v in shuffled_values:               # ...vs. inserting in random order
    balanced_ish_root = bst_insert(balanced_ish_root, v)

print("height after sorted-order inserts:  ", tree_height(degenerate_root))
print("height after shuffled-order inserts:", tree_height(balanced_ish_root))
assert tree_height(degenerate_root) == n - 1        # a straight line: worst case
assert tree_height(balanced_ish_root) < n // 2       # much shorter, typically ~log2(n)

# Time searching for the last-inserted (worst-case) value in each tree
degenerate_time = timeit.timeit(
    lambda: bst_search(degenerate_root, n - 1), number=2000
)
balanced_time = timeit.timeit(
    lambda: bst_search(balanced_ish_root, shuffled_values[-1]), number=2000
)
print(f"search time, degenerate tree:   {degenerate_time:.4f}s")
print(f"search time, shuffled tree:     {balanced_time:.4f}s")
assert degenerate_time > balanced_time  # the O(n) tree is measurably slower
print("Degeneration demonstrated: sorted-order inserts produce an O(n) tree")

## 6. Keeping trees balanced: AVL rotations

The degeneration above is exactly the problem self-balancing trees exist to
solve. An AVL tree adds one rule on top of the BST property: for every
node, the heights of its left and right subtrees may differ by at most 1
(the "balance factor" is in {-1, 0, 1}). After every insertion, the tree
walks back up from the new node to the root, and the moment any node's
balance factor goes outside that range, it performs a *rotation* — a local
rewiring of a small number of pointers — to restore balance without
disturbing the BST ordering.

There are two single rotations (left-left and right-right cases, mirror
images of each other) and two double rotations (left-right and right-left,
which are really just two single rotations composed). You do not need to
memorize all four independently: once you understand `rotate_left` and
`rotate_right` as pure pointer surgery, the four "cases" are just about
figuring out *which* rotation, or pair of rotations, restores balance at a
given node. Red-black trees solve the same problem with a different
mechanism — nodes are colored red or black, and a small set of coloring and
rotation rules bounds the height at roughly 2·log2(n) instead of AVL's
tighter ~1.44·log2(n); red-black trees do fewer rotations per insertion on
average, which is why they are the more common choice inside language
standard libraries (for example, C++'s `std::map`), even though AVL trees
are, height for height, slightly more rigidly balanced.

In [ ]:
class AVLNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None
        self.height = 0   # height of the subtree rooted here; a leaf has height 0


def h(node):
    """Height helper that treats an empty subtree (None) as height -1."""
    return node.height if node is not None else -1


def update_height(node):
    node.height = 1 + max(h(node.left), h(node.right))


def balance_factor(node):
    return h(node.left) - h(node.right)   # positive = left-heavy, negative = right-heavy


def rotate_left(node):
    """Right-heavy node becomes the LEFT child of what was its right child."""
    new_root = node.right
    node.right = new_root.left   # new_root's old left subtree moves under node
    new_root.left = node         # node drops down to become new_root's left child
    update_height(node)          # heights must be recomputed bottom-up: node first...
    update_height(new_root)      # ...then new_root, since it now depends on node
    return new_root


def rotate_right(node):
    """Left-heavy node becomes the RIGHT child of what was its left child."""
    new_root = node.left
    node.left = new_root.right
    new_root.right = node
    update_height(node)
    update_height(new_root)
    return new_root


def avl_insert(node, value):
    """Standard BST insert, then rebalance on the way back up the recursion."""
    if node is None:
        return AVLNode(value)
    if value < node.value:
        node.left = avl_insert(node.left, value)
    elif value > node.value:
        node.right = avl_insert(node.right, value)
    else:
        return node                       # no duplicates

    update_height(node)
    bf = balance_factor(node)

    if bf > 1:                            # left-heavy: needs a right rotation
        if balance_factor(node.left) < 0:
            node.left = rotate_left(node.left)   # left-right case: fix left child first
        return rotate_right(node)                 # left-left case (or left-right, now fixed)

    if bf < -1:                           # right-heavy: needs a left rotation
        if balance_factor(node.right) > 0:
            node.right = rotate_right(node.right)  # right-left case: fix right child first
        return rotate_left(node)                    # right-right case (or right-left, now fixed)

    return node                           # already balanced, nothing to do


avl_root = None
for v in [1, 2, 3, 4, 5, 6, 7]:   # sorted input: exactly the degenerate case from Section 5
    avl_root = avl_insert(avl_root, v)

plain_root = None
for v in [1, 2, 3, 4, 5, 6, 7]:
    plain_root = bst_insert(plain_root, v)

print("plain BST height after sorted inserts:", tree_height(plain_root))   # degenerate
print("AVL tree height after sorted inserts: ", avl_root.height)           # kept short
assert tree_height(plain_root) == 6     # a straight line of 7 nodes
assert avl_root.height == 2             # AVL keeps log2(7) ≈ 2.8 -> height 2
print("AVL rotations kept the tree balanced on the exact input that broke the plain BST")

## 7. Ordered-set operations: range query, floor, and ceiling

The BST property gives you more than just search — because inorder
traversal visits values in sorted order, a BST is really an *ordered set*,
and that unlocks queries that a hash set cannot answer efficiently: "give me
everything between 3 and 9", or "what's the largest value not exceeding 7"
(`floor`), or "what's the smallest value not less than 7" (`ceiling`). Each
of these still runs in roughly O(h) or O(h + k) time (k = number of results
returned), because the BST property lets you prune whole subtrees that
cannot possibly contain a qualifying value, exactly as in search.

`range_query` is the one to read carefully: it only recurses left when the
current node's value is greater than `low` (there could be qualifying values
on the left), and only recurses right when it is less than `high` (same
logic, mirrored) — pruning exactly the subtrees that are provably out of
range rather than visiting everything and filtering afterward.

In [ ]:
def range_query(node, low, high, result=None):
    """Collect all values v with low <= v <= high, in sorted order."""
    if result is None:
        result = []
    if node is None:
        return result
    if node.value > low:                       # only bother with left if it could help
        range_query(node.left, low, high, result)
    if low <= node.value <= high:
        result.append(node.value)
    if node.value < high:                       # only bother with right if it could help
        range_query(node.right, low, high, result)
    return result


def floor(node, target):
    """Largest value in the tree that is <= target, or None if none exists."""
    if node is None:
        return None
    if node.value == target:
        return node.value
    if node.value > target:
        return floor(node.left, target)          # everything here is too big; go left
    right_floor = floor(node.right, target)       # try to do better on the right...
    return right_floor if right_floor is not None else node.value  # ...else this node is best


def ceiling(node, target):
    """Smallest value in the tree that is >= target, or None if none exists."""
    if node is None:
        return None
    if node.value == target:
        return node.value
    if node.value < target:
        return ceiling(node.right, target)
    left_ceiling = ceiling(node.left, target)
    return left_ceiling if left_ceiling is not None else node.value


assert range_query(root, 3, 9) == [3, 6, 8]
assert floor(root, 7) == 6
assert ceiling(root, 7) == 8
print("range_query(3, 9):", range_query(root, 3, 9))
print("floor(7):  ", floor(root, 7))
print("ceiling(7):", ceiling(root, 7))
print("Range query, floor, and ceiling checks passed")

### A free sorting algorithm: treesort

Because inorder traversal of a BST produces sorted output, building a BST
from a list and then reading it back with `inorder_recursive` is a valid
(if unusual) sorting algorithm, called treesort. It is worth seeing once,
but it is rarely used in practice: its worst case is exactly the
degeneration from Section 5 (sorted or adversarial input gives O(n) insert
per element, O(n²) overall), and even in the average case it uses more
memory and has worse constants than a well-tuned comparison sort like
Python's built-in Timsort.

In [ ]:
def treesort(values):
    t_root = None
    for value in values:
        t_root = bst_insert(t_root, value)
    return inorder_recursive(t_root)


assert treesort([5, 2, 8, 1, 9]) == [1, 2, 5, 8, 9]
print("treesort([5, 2, 8, 1, 9]) ->", treesort([5, 2, 8, 1, 9]))
print("Treesort check passed")

## 8. B-trees: multi-way branching for disk-optimized storage

Every tree so far — the plain BST, the AVL tree — is designed to live
entirely in memory: a node is a small object with a couple of pointers, and
following a pointer costs essentially nothing because RAM access is
uniformly fast. Databases and filesystems face a very different cost model.
Their trees are usually far too large to fit in memory, so most nodes live
on disk, and reading a node means an actual disk I/O — many orders of
magnitude slower than a RAM access, and, critically, it happens in
fixed-size chunks called blocks (typically 4 KB or more) regardless of how
much of the block you actually need. A binary tree node holding one key
wastes almost an entire disk block on padding; worse, a binary tree of n
keys has height around log2(n), so a lookup costs log2(n) separate block
reads. A B-tree attacks exactly this problem: instead of one key and two
children per node, a B-tree node holds many keys and many children — as
many as comfortably fit in one disk block — so a single I/O reads dozens or
hundreds of keys at once, and the height of the tree (and therefore the
number of I/Os per lookup) drops to roughly log_m(n) for a branching factor
m in the hundreds. This is why B-trees (and their descendant B+-trees) are
the workhorse index structure inside essentially every relational database
and filesystem, while AVL and red-black trees remain the workhorse choice
for in-memory ordered structures such as C++'s `std::map` or Java's
`TreeMap`. The other structural difference follows from the same
disk-oriented design: AVL and red-black trees stay balanced by rotating
pointers after each insertion, which can leave subtrees at different
depths from node to node (bounded, but not equal); a B-tree instead grows
and shrinks from the root, splitting or merging whole nodes, which keeps
every leaf at exactly the same depth by construction — there is no
analogue of a rotation, and no possibility of the tree becoming lopsided
the way a plain BST can.

The examples below use an order-4 B-tree, meaning each node holds at
most 3 keys and at most 4 children (in general, an order-m B-tree node
holds at most m - 1 keys and at most m children). Within a node, the keys
are kept sorted, and they divide the node's children into ranges: with
keys `[k1, k2, k3]`, the first child holds everything less than `k1`, the
second child holds everything between `k1` and `k2`, and so on — exactly
one more child than there are keys. Every node except the root must hold
at least `ceil(m / 2) - 1` keys (so at least 1 key for order 4), which
keeps nodes from becoming nearly empty; leaves hold no children at all,
only keys. The defining shape guarantee is that every leaf sits at the
same depth: because the tree only grows by splitting a full node (which
can push a new root into existence) rather than by attaching a new leaf
somewhere deep in the middle, there is never a way for one branch to end
up taller than another.

In [ ]:
class BTreeNode:
    """A single B-tree node: sorted keys, plus one more child than keys
    (leaves hold no children at all).
    """

    def __init__(self, leaf=True):
        self.leaf = leaf
        self.keys = []       # sorted list of keys stored in this node
        self.children = []   # child BTreeNode references; empty for a leaf

    def __repr__(self):
        return f"BTreeNode(keys={self.keys}, leaf={self.leaf})"


ORDER = 4                  # each node holds at most ORDER children
MAX_KEYS = ORDER - 1       # = 3 for this order-4 B-tree

Insertion always starts the way a BST insertion does: walk down from
the root, using the sorted keys at each node to choose which child to
descend into, until you reach a leaf, and insert the new key there in
sorted position. The B-tree-specific step happens afterward: if that
insertion pushes a node's key count past `MAX_KEYS`, the node has
overflowed and must split. Splitting an overfull node is straightforward:
find its median key, move every key to the median's left into a new left
node and every key to its right into a new right node, and promote the
median key one level up into the parent (along with a pointer to the new
right node). If the parent itself now has too many keys, it overflows in
turn and splits the same way — the overflow can propagate all the way up
to the root, and if even the root splits, a brand-new root is created
holding just the promoted median, which is the only way a B-tree ever
grows taller. This "split on the way back up" is directly analogous to
"rotate on the way back up" in an AVL tree: both are how the structure
repairs itself after an insertion, just with node-splitting instead of
pointer rotation.

Deletion is more involved and is out of scope for this course: removing a
key can leave a node under the minimum key count, which then has to
*borrow* a key from an adjacent sibling (if that sibling has one to
spare) or *merge* with a sibling (pulling a key down from the parent), and
that merging can itself propagate underflow up toward the root —
conceptually the mirror image of splitting, but with more cases to
handle. The implementation below therefore only covers search and
insertion, which is enough to see how the balance guarantee is
maintained.

In [ ]:
def _split_child(node):
    """Split an overfull node in half; return (median_key, new_right_node)
    so the caller can promote the median into the parent.
    """
    mid = len(node.keys) // 2
    median = node.keys[mid]

    right = BTreeNode(leaf=node.leaf)
    right.keys = node.keys[mid + 1:]
    node.keys = node.keys[:mid]

    if not node.leaf:
        right.children = node.children[mid + 1:]
        node.children = node.children[:mid + 1]

    return median, right


def _insert_into_node(node, key, max_keys):
    """Insert key into the subtree rooted at node; return (median, right)
    if node itself overflowed and had to split, else None.
    """
    if node.leaf:
        i = 0
        while i < len(node.keys) and key > node.keys[i]:
            i += 1
        node.keys.insert(i, key)
    else:
        i = 0
        while i < len(node.keys) and key > node.keys[i]:
            i += 1
        result = _insert_into_node(node.children[i], key, max_keys)
        if result is not None:
            median, right_child = result
            node.keys.insert(i, median)              # promoted median lands here
            node.children.insert(i + 1, right_child)  # new right subtree goes next to it

    if len(node.keys) > max_keys:      # this node overflowed; split it
        return _split_child(node)
    return None


def btree_insert(root, key, max_keys=MAX_KEYS):
    """Insert key into the order-4 B-tree rooted at root; return the
    (possibly new) root.
    """
    if root is None:
        root = BTreeNode(leaf=True)

    result = _insert_into_node(root, key, max_keys)
    if result is not None:                 # the root itself overflowed
        median, right_child = result
        new_root = BTreeNode(leaf=False)
        new_root.keys = [median]
        new_root.children = [root, right_child]
        return new_root

    return root


def print_btree(root, level=0):
    """Print each level of the B-tree indented by depth, for inspection."""
    if root is None:
        print("  " * level + "(empty)")
        return
    print("  " * level + str(root.keys))
    for child in root.children:
        print_btree(child, level + 1)


def inorder_keys(node, result=None):
    """Return every key in the B-tree rooted at node, in sorted order."""
    if result is None:
        result = []
    if node is None:
        return result
    if node.leaf:
        result.extend(node.keys)
        return result
    for i, key in enumerate(node.keys):
        inorder_keys(node.children[i], result)
        result.append(key)
    inorder_keys(node.children[-1], result)
    return result

### Search, and a worked example of a split

Search in a B-tree is simpler than insertion: at each node, scan its
sorted keys to find either an exact match or the correct child to descend
into (the same "which range does this key fall in" logic already used
above to route an insertion to the right leaf), and repeat until you find
the key or reach a leaf without it. You will implement `btree_search`
yourself in this section's exercise below, using `order4_root` — the tree
built by the worked example that follows — as the test case.

The insertion sequence below is chosen to trigger two splits: an early
root split, when the very first node overflows and the tree grows a
second level, and a later split of an internal leaf once it fills back up.
Watch the shape of the tree printed after each insertion — in particular,
watch every leaf stay at exactly the same depth even as the tree grows.

In [ ]:
order4_root = None
insertion_sequence = [10, 20, 30, 15, 25, 5, 35, 1, 40]

for key in insertion_sequence:
    print(f"--- inserting {key} ---")
    print("before:")
    print_btree(order4_root)
    order4_root = btree_insert(order4_root, key, max_keys=MAX_KEYS)
    print("after:")
    print_btree(order4_root)
    print()

assert inorder_keys(order4_root) == sorted(insertion_sequence)
print("Final tree holds all keys in sorted order:", inorder_keys(order4_root))

As the printout above shows, inserting `15` overflows the root (which
had grown to `[10, 20, 30]`, one key past `MAX_KEYS`): the root splits into
two leaves under a brand-new root holding just the median, `20`, and the
tree grows from one level to two. Later, inserting `1` overflows the
left-most leaf (`[1, 5, 10, 15]`) the same way, but this split does not
reach the root — the promoted median, `10`, fits into the existing root
without overflowing it, so the tree stays at two levels. Both are the same
split-on-overflow mechanic; the only difference is how far the overflow
propagates.

## Exercises

Complete each function below, then run its self-check cell. Try to solve
each one before looking at the `## Solutions` section at the end.

### Exercise 1 — Count the leaves

Write `count_leaves(node)`, which returns the number of leaf nodes (nodes
with no children) in the tree rooted at `node`.

Example: for the tree built in Section 1 (root 8, with leaves 1, 6, 14),
`count_leaves(root)` should return `3`.

In [ ]:
def count_leaves(node):
    """Return the number of leaf nodes (no left AND no right child) in the tree.

    A leaf is a node whose left and right children are both None.
    An empty tree (node is None) has 0 leaves.
    """
    # TODO: implement this
    raise NotImplementedError

In [ ]:
# Self-check for Exercise 1
assert count_leaves(None) == 0
assert count_leaves(TreeNode(42)) == 1          # a single node is its own leaf
assert count_leaves(root) == 3                  # leaves 1, 6, 14 in the Section 1 tree
small = TreeNode(1)
small.left = TreeNode(2)
assert count_leaves(small) == 1                 # only node 2 is a leaf
print("\u2705 Exercise 1 passed")

### Exercise 2 — Mirror a tree

Write `mirror(node)`, which returns a **new** tree that is the left-right
mirror image of the tree rooted at `node` (every node's left and right
children are swapped, recursively). Do not mutate the original tree.

Example: mirroring
```
    8            8
   / \    ->    / \
  3  10        10   3
```
so that `inorder_recursive(mirror(root))` is the reverse of
`inorder_recursive(root)`.

In [ ]:
def mirror(node):
    """Return a NEW tree that is the mirror image of the tree rooted at node.

    The original tree (node and its descendants) must be left unchanged.
    An empty tree mirrors to an empty tree (None).
    """
    # TODO: implement this
    raise NotImplementedError

In [ ]:
# Self-check for Exercise 2
assert mirror(None) is None
mirrored = mirror(root)
assert inorder_recursive(mirrored) == list(reversed(inorder_recursive(root)))
assert mirrored.value == 8
assert mirrored.left.value == 10   # original right child is now the left child
assert mirrored.right.value == 3   # original left child is now the right child
# original tree must be unchanged
assert root.left.value == 3
assert root.right.value == 10
print("\u2705 Exercise 2 passed")

### Exercise 3 — Lowest common ancestor in a BST

Write `bst_lowest_common_ancestor(node, value_a, value_b)`, which returns
the value of the deepest node in a **binary search tree** that has both
`value_a` and `value_b` in its subtree (a node counts as its own ancestor).
You may assume both values exist in the tree.

Hint: use the BST property. If both values are less than the current node,
the answer is somewhere in the left subtree; if both are greater, it's in
the right subtree; otherwise the current node is the split point, and the
answer.

Example: for the BST built in Section 4 (values 1, 3, 4, 5, 7, 8, 9),
`bst_lowest_common_ancestor(bst_root, 1, 4)` should return `3`.

In [ ]:
def bst_lowest_common_ancestor(node, value_a, value_b):
    """Return the value of the lowest common ancestor of value_a and value_b
    in the binary search tree rooted at node. Both values are assumed present.
    """
    # TODO: implement this
    raise NotImplementedError

In [ ]:
# Self-check for Exercise 3
fresh_bst = None
for v in [5, 3, 8, 1, 4, 7, 9]:
    fresh_bst = bst_insert(fresh_bst, v)

assert bst_lowest_common_ancestor(fresh_bst, 1, 4) == 3
assert bst_lowest_common_ancestor(fresh_bst, 7, 9) == 8
assert bst_lowest_common_ancestor(fresh_bst, 1, 9) == 5   # root is the split point
assert bst_lowest_common_ancestor(fresh_bst, 4, 4) == 4   # a value with itself
print("\u2705 Exercise 3 passed")

### Exercise 4 — Serialize and deserialize a binary tree

Write two functions:

- `serialize(node)`: return a list of values representing the tree using
  preorder traversal, using the sentinel `None` wherever a child is missing
  (so the shape of the tree, not just a BST-recoverable value set, is
  preserved — this works for *any* binary tree, not just BSTs).
- `deserialize(values)`: given a list produced by `serialize`, rebuild and
  return the root `TreeNode` of an equivalent tree.

Example: for a tree with only a root of value 1 and a right child of value
2, `serialize` should return `[1, None, 2, None, None]` (root, then its
empty left, then its right subtree serialized the same way).

This is harder than the previous three exercises: think about consuming the
list from the front (an iterator, or an index that both functions share, is
easier to manage than slicing).

In [ ]:
def serialize(node):
    """Preorder-serialize node into a flat list, using None for missing children."""
    # TODO: implement this
    raise NotImplementedError


def deserialize(values):
    """Rebuild a tree from a list produced by serialize(), returning its root."""
    # TODO: implement this
    raise NotImplementedError

In [ ]:
# Self-check for Exercise 4
tiny = TreeNode(1)
tiny.right = TreeNode(2)
assert serialize(tiny) == [1, None, 2, None, None]

rebuilt_tiny = deserialize(serialize(tiny))
assert rebuilt_tiny.value == 1
assert rebuilt_tiny.left is None
assert rebuilt_tiny.right.value == 2

# Round-trip the Section 1 tree through serialize/deserialize
rebuilt_root = deserialize(serialize(root))
assert preorder_recursive(rebuilt_root) == preorder_recursive(root)
assert inorder_recursive(rebuilt_root) == inorder_recursive(root)
assert postorder_recursive(rebuilt_root) == postorder_recursive(root)
print("\u2705 Exercise 4 passed")

### Exercise 5 — B-tree search

Write `btree_search(node, key, max_keys=MAX_KEYS)`, which returns the
`BTreeNode` that contains `key`, or `None` if `key` is not present
anywhere in the B-tree rooted at `node`. At each node, scan its sorted
`keys` list to find either an exact match or the index of the child whose
range covers `key`, and descend into that child; if `node` is a leaf and
no match was found, `key` is not present in the tree.

Test it against `order4_root`, the order-4 B-tree built in Section 8.

**5. Why do B-trees use many keys per node (multi-way branching) instead
of the binary branching used by AVL and red-black trees, and where are
B-trees typically used as a result?**

<details><summary>Show answer</summary>
B-trees are designed around the cost of disk I/O rather than the cost of
following an in-memory pointer: reading anything from disk pulls in a
whole fixed-size block regardless of how much of it is used, so a node
that holds many keys (sized to fill one block) turns a single disk read
into dozens or hundreds of key comparisons, and it also shrinks the tree's
height (and therefore the number of disk reads per lookup) to roughly
log_m(n) for a large branching factor m. A binary node would waste most of
each block and force one disk read per level of a much taller tree. This
is why B-trees (and B+-trees) are the standard index structure inside
relational databases and filesystems, while AVL and red-black trees, whose
per-node pointer-chasing cost is negligible in RAM, remain the standard
choice for in-memory ordered structures.
</details>

**6. Unlike an AVL tree, a B-tree never performs a rotation, yet every
leaf stays at exactly the same depth. How does a B-tree keep itself
balanced without rotations?**

<details><summary>Show answer</summary>
A B-tree balances itself by growing and shrinking from the root instead of
rewiring pointers locally. On insertion, a node that overflows past its
maximum key count splits in half and promotes its median key to its
parent; if that overflow propagates all the way to the root, the root
splits too and a brand-new root is created above it, which is the only way
the tree gains a level. Because every leaf is pushed down together by a
root split (never individually, the way a BST attaches one new leaf deep
in a single branch), every leaf ends up at the same depth by construction.
Deletion works the same way in reverse: an under-full node borrows from or
merges with a sibling, and merging can propagate underflow up toward the
root, shrinking the tree's height only at the root when needed.
</details>

**5. Why do B-trees use many keys per node (multi-way branching) instead
of the binary branching used by AVL and red-black trees, and where are
B-trees typically used as a result?**

<details><summary>Show answer</summary>
B-trees are designed around the cost of disk I/O rather than the cost of
following an in-memory pointer: reading anything from disk pulls in a
whole fixed-size block regardless of how much of it is used, so a node
that holds many keys (sized to fill one block) turns a single disk read
into dozens or hundreds of key comparisons, and it also shrinks the tree's
height (and therefore the number of disk reads per lookup) to roughly
log_m(n) for a large branching factor m. A binary node would waste most of
each block and force one disk read per level of a much taller tree. This
is why B-trees (and B+-trees) are the standard index structure inside
relational databases and filesystems, while AVL and red-black trees, whose
per-node pointer-chasing cost is negligible in RAM, remain the standard
choice for in-memory ordered structures.
</details>

**6. Unlike an AVL tree, a B-tree never performs a rotation, yet every
leaf stays at exactly the same depth. How does a B-tree keep itself
balanced without rotations?**

<details><summary>Show answer</summary>
A B-tree balances itself by growing and shrinking from the root instead of
rewiring pointers locally. On insertion, a node that overflows past its
maximum key count splits in half and promotes its median key to its
parent; if that overflow propagates all the way to the root, the root
splits too and a brand-new root is created above it, which is the only way
the tree gains a level. Because every leaf is pushed down together by a
root split (never individually, the way a BST attaches one new leaf deep
in a single branch), every leaf ends up at the same depth by construction.
Deletion works the same way in reverse: an under-full node borrows from or
merges with a sibling, and merging can propagate underflow up toward the
root, shrinking the tree's height only at the root when needed.
</details>

In [ ]:
def btree_search(node, key, max_keys=MAX_KEYS):
    """Return the BTreeNode containing key, or None if key is not present.

    Scan the sorted keys at each node to find an exact match or the child
    whose range covers key, and descend; a leaf with no match means key
    is absent from the tree.
    """
    # TODO: implement this
    raise NotImplementedError

In [ ]:
# Self-check for Exercise 5
for key in insertion_sequence:
    found_node = btree_search(order4_root, key)
    assert found_node is not None
    assert key in found_node.keys

assert btree_search(order4_root, 999) is None    # not in the tree
assert btree_search(None, 5) is None              # empty tree

root_hit = btree_search(order4_root, order4_root.keys[0])
assert root_hit is order4_root       # a key stored in the root is found there directly
print("Exercise 5 checks passed")

## Quiz

Try to answer each question yourself before revealing the answer.

**1. What is the time complexity of `bst_search` on a balanced BST with n
nodes? What about on a completely degenerate BST (built by inserting
already-sorted data)?**

<details><summary>Show answer</summary>
O(log n) on a balanced BST, because the height h is about log2(n) and each
step of the search discards one whole subtree. On a degenerate BST the
height is n - 1, so search is O(n) — no better than scanning a linked list.
</details>

**2. Why does inorder traversal of a BST produce values in sorted order,
while preorder and postorder do not?**

<details><summary>Show answer</summary>
Inorder visits left subtree, then the node, then right subtree. By the BST
property everything in the left subtree is smaller than the node and
everything in the right subtree is larger, so visiting left-node-right at
every level recursively produces values from smallest to largest. Preorder
and postorder visit the node itself either before or after both subtrees
are fully processed, which does not respect the smaller/larger split at
every level, so there is no reason for their output to be sorted.
</details>

**3. When deleting a node with two children from a BST, why do we replace
its value with the in-order successor's value instead of just removing the
node directly?**

<details><summary>Show answer</summary>
Removing the node directly would disconnect both of its subtrees from the
tree, since a node has only one parent link pointing to it. The in-order
successor (the smallest value in the right subtree) is the next value in
sorted order after the deleted node, so putting it in the deleted node's
place preserves the BST property everywhere; and the successor itself is
guaranteed to have no left child (it is the leftmost node of the right
subtree), so deleting it afterward is only ever a leaf or one-child case,
never another two-children case.
</details>

**4. An AVL tree and a red-black tree both guarantee O(log n) height for
n nodes. What is the practical trade-off between them?**

<details><summary>Show answer</summary>
AVL trees enforce a tighter balance condition (subtree heights differ by at
most 1 everywhere), which keeps their height closer to the theoretical
minimum log2(n) and makes lookups marginally faster. Red-black trees allow
more slack (height bounded by roughly 2·log2(n)) in exchange for needing
fewer rotations per insertion or deletion on average, which makes them
faster to keep balanced under frequent modification. This is why many
language standard libraries (for example, C++'s `std::map`) use red-black
trees rather than AVL trees.
</details>

## Solutions (try the exercises yourself first!)

Fully worked solutions to all four exercises follow. Compare against your
own implementation rather than just reading these — the point is the
practice, not the answer.

In [ ]:
# Solution 1 — Count the leaves
def count_leaves_solution(node):
    if node is None:
        return 0
    if node.left is None and node.right is None:
        return 1
    return count_leaves_solution(node.left) + count_leaves_solution(node.right)


assert count_leaves_solution(None) == 0
assert count_leaves_solution(root) == 3
print("Solution 1 verified")

In [ ]:
# Solution 2 — Mirror a tree
def mirror_solution(node):
    if node is None:
        return None
    mirrored_node = TreeNode(node.value)
    mirrored_node.left = mirror_solution(node.right)   # old right becomes new left
    mirrored_node.right = mirror_solution(node.left)   # old left becomes new right
    return mirrored_node


mirrored_check = mirror_solution(root)
assert inorder_recursive(mirrored_check) == list(reversed(inorder_recursive(root)))
print("Solution 2 verified")

In [ ]:
# Solution 3 — Lowest common ancestor in a BST
def bst_lca_solution(node, value_a, value_b):
    if value_a < node.value and value_b < node.value:
        return bst_lca_solution(node.left, value_a, value_b)
    if value_a > node.value and value_b > node.value:
        return bst_lca_solution(node.right, value_a, value_b)
    return node.value   # values split here (or one equals node.value): this is the LCA


fresh_bst_check = None
for v in [5, 3, 8, 1, 4, 7, 9]:
    fresh_bst_check = bst_insert(fresh_bst_check, v)
assert bst_lca_solution(fresh_bst_check, 1, 4) == 3
assert bst_lca_solution(fresh_bst_check, 7, 9) == 8
print("Solution 3 verified")

In [ ]:
# Solution 4 — Serialize and deserialize a binary tree
def serialize_solution(node):
    result = []

    def visit(n):
        if n is None:
            result.append(None)
            return
        result.append(n.value)
        visit(n.left)
        visit(n.right)

    visit(node)
    return result


def deserialize_solution(values):
    it = iter(values)

    def build():
        value = next(it)
        if value is None:
            return None
        n = TreeNode(value)
        n.left = build()
        n.right = build()
        return n

    return build()


tiny_check = TreeNode(1)
tiny_check.right = TreeNode(2)
assert serialize_solution(tiny_check) == [1, None, 2, None, None]
rebuilt_check = deserialize_solution(serialize_solution(root))
assert preorder_recursive(rebuilt_check) == preorder_recursive(root)
print("Solution 4 verified")

In [ ]:
# Solution 5 — B-tree search
def btree_search_solution(node, key, max_keys=MAX_KEYS):
    if node is None:
        return None
    i = 0
    while i < len(node.keys) and key > node.keys[i]:
        i += 1
    if i < len(node.keys) and key == node.keys[i]:
        return node
    if node.leaf:
        return None
    return btree_search_solution(node.children[i], key, max_keys)


for key in insertion_sequence:
    found_node = btree_search_solution(order4_root, key)
    assert found_node is not None
    assert key in found_node.keys
assert btree_search_solution(order4_root, 999) is None
print("Solution 5 verified")

## MTech Extension — augmented trees and order statistics

Everything above treats a BST node as holding only a value. An *augmented*
BST stores one extra piece of derived information per node — here, the size
of the subtree rooted at that node — and maintains it incrementally on every
insert. That small addition upgrades the tree from an ordered set into an
**order-statistics tree**: it can answer "what is the k-th smallest value?"
and "what is the rank of value v?" in O(h) time each, instead of the O(n)
an unaugmented tree would need (walk the whole thing and count).

The general pattern — augment a node with a value that can be computed from
its children's augmented values, and update it in O(1) per node on the way
back up the same recursion that already does insertion — is the same
technique used for interval trees, order-statistics trees in the Linux
kernel's red-black tree implementation, and segment trees more broadly. The
key discipline is that the augmented field must be correct at every
ancestor after every mutation, not just at the node that changed, which is
why `update_size` below is called on the way back up `augmented_insert`
rather than computed lazily on demand.

In [ ]:
class AugmentedNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None
        self.size = 1   # number of nodes in the subtree rooted here, including itself


def subtree_size(node):
    return node.size if node is not None else 0


def update_size(node):
    node.size = 1 + subtree_size(node.left) + subtree_size(node.right)


def augmented_insert(node, value):
    if node is None:
        return AugmentedNode(value)
    if value < node.value:
        node.left = augmented_insert(node.left, value)
    elif value > node.value:
        node.right = augmented_insert(node.right, value)
    else:
        return node
    update_size(node)          # recompute AFTER the recursive call returns:
    return node                # children's sizes must already be up to date


def select(node, k):
    """Return the value with rank k (0-indexed, smallest = rank 0) in the subtree."""
    if node is None:
        raise IndexError("rank out of range")
    left_size = subtree_size(node.left)
    if k < left_size:
        return select(node.left, k)          # the k-th smallest is in the left subtree
    if k == left_size:
        return node.value                    # this node IS the k-th smallest
    return select(node.right, k - left_size - 1)  # skip left subtree and this node


def rank(node, value):
    """Return the number of values in the tree strictly less than value."""
    if node is None:
        return 0
    if value <= node.value:
        return rank(node.left, value)
    return subtree_size(node.left) + 1 + rank(node.right, value)


augmented_root = None
for v in [5, 3, 8, 1, 4, 7, 9]:
    augmented_root = augmented_insert(augmented_root, v)

sorted_values = [1, 3, 4, 5, 7, 8, 9]
for k, expected in enumerate(sorted_values):
    assert select(augmented_root, k) == expected   # select(k) matches sorted position k

assert rank(augmented_root, 5) == 3    # three values (1, 3, 4) are strictly less than 5
assert rank(augmented_root, 1) == 0    # nothing is less than the minimum
assert augmented_root.size == 7

print("select(3) [4th smallest]:", select(augmented_root, 3))
print("rank(7) [values < 7]:    ", rank(augmented_root, 7))
print("Order-statistics augmentation checks passed: select and rank both O(h)")

**Discussion point for MTech:** `select` and `rank` above are each O(h),
exactly like `bst_search` — the augmentation adds O(1) work per node
visited, not an extra pass over the tree. But this only holds if `size` is
kept correct through *every* mutation, including deletion (not implemented
above): an augmented deletion must decrement `size` on every ancestor on the
path back to the root, mirroring `update_size`'s role in `augmented_insert`.
A tree where `size` is ever allowed to drift out of sync with reality is
worse than useless — it returns wrong answers silently rather than raising
an error, which is a much harder bug to catch in production.